In [6]:
# ============================
# RetailPulse - Section 4 (Final Corrected)
# Customer Churn Prediction
# ============================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# ----------------------------
# Step 1: Load cleaned dataset
# ----------------------------
df = pd.read_csv("../data/processed/cleaned_retail.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# ----------------------------
# Step 2: Aggregate to Customer Level
# ----------------------------
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

customer_df = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,   # Recency
    'Invoice': 'count',                                       # Frequency
    'TotalPrice': 'sum'                                       # Monetary
}).rename(columns={'InvoiceDate':'Recency','Invoice':'Frequency','TotalPrice':'Monetary'}).reset_index()

# ----------------------------
# Step 3: Define Churn Label
# ----------------------------
customer_df['Churn'] = (customer_df['Recency'] > 90).astype(int)

print("Churn distribution:\n", customer_df['Churn'].value_counts())

# ----------------------------
# Step 4: Prepare Features (exclude Recency to avoid leakage)
# ----------------------------
X = customer_df[['Frequency','Monetary']]
y = customer_df['Churn']

# Stratified random split ensures both classes are present
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ----------------------------
# Step 5: Train Models
# ----------------------------
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
log_preds = log_model.predict(X_test)

rf_model = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

gb_model = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)

# ----------------------------
# Step 6: Evaluation
# ----------------------------
def evaluate(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    print(f"\n{model_name}")
    print(f"Accuracy : {acc:.2f}")
    print(f"Precision: {prec:.2f}")
    print(f"Recall   : {rec:.2f}")
    print(f"F1 Score : {f1:.2f}")
    print(f"ROC AUC  : {auc:.2f}")
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

evaluate(y_test, log_preds, "Logistic Regression")
evaluate(y_test, rf_preds, "Random Forest")
evaluate(y_test, gb_preds, "Gradient Boosting")

# ----------------------------
# Step 7: Precision@Top20%
# ----------------------------
import numpy as np
from sklearn.metrics import precision_score

def precision_at_k(y_true, y_scores, k=0.2):
    cutoff = int(len(y_scores) * k)
    top_idx = np.argsort(y_scores)[-cutoff:]
    return precision_score(y_true.iloc[top_idx], np.ones(len(top_idx)))

gb_scores = gb_model.predict_proba(X_test)[:,1]
print("\nPrecision@Top20% (Gradient Boosting):", precision_at_k(y_test, gb_scores))


Churn distribution:
 Churn
1    3024
0    2918
Name: count, dtype: int64

Logistic Regression
Accuracy : 0.66
Precision: 0.62
Recall   : 0.86
F1 Score : 0.72
ROC AUC  : 0.66
Confusion Matrix:
 [[266 318]
 [ 87 518]]

Random Forest
Accuracy : 0.66
Precision: 0.65
Recall   : 0.74
F1 Score : 0.69
ROC AUC  : 0.66
Confusion Matrix:
 [[339 245]
 [155 450]]

Gradient Boosting
Accuracy : 0.67
Precision: 0.65
Recall   : 0.74
F1 Score : 0.69
ROC AUC  : 0.66
Confusion Matrix:
 [[345 239]
 [158 447]]

Precision@Top20% (Gradient Boosting): 0.7721518987341772
